In [1]:
# =========================================
# C1. IMPORTS
# =========================================
import json
import pickle
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# =========================================
# C2. PATHS
# =========================================
DATA_DIR = Path(r"C:\D drive\item_recommendation\data")

MODEL_DIR = DATA_DIR / "model_outputs"
ARTIFACT_DIR = DATA_DIR / "stage1_artifacts"

MODEL_FILE = MODEL_DIR / "xgboost_ranker_model.json"
FEATURE_FILE = MODEL_DIR / "ranker_feature_columns.json"

print("Model exists:", MODEL_FILE.exists())
print("Feature file exists:", FEATURE_FILE.exists())

Model exists: True
Feature file exists: True


In [3]:
# =========================================
# C3. LOAD MODEL AND ARTIFACTS
# =========================================
ranker = xgb.XGBRanker()
ranker.load_model(MODEL_FILE)

with open(FEATURE_FILE, "r", encoding="utf-8") as f:
    trained_feature_columns = json.load(f)

with open(ARTIFACT_DIR / "association_rules.pkl", "rb") as f:
    rules_df = pickle.load(f)

with open(ARTIFACT_DIR / "item_pair_counter.pkl", "rb") as f:
    item_pair_counter = pickle.load(f)

with open(ARTIFACT_DIR / "context_item_counter.pkl", "rb") as f:
    context_item_counter = pickle.load(f)

with open(ARTIFACT_DIR / "user_item_counter.pkl", "rb") as f:
    user_item_counter = pickle.load(f)

with open(ARTIFACT_DIR / "user_category_counter.pkl", "rb") as f:
    user_category_counter = pickle.load(f)

with open(ARTIFACT_DIR / "item_to_category.pkl", "rb") as f:
    item_to_category = pickle.load(f)

with open(ARTIFACT_DIR / "item_to_name.pkl", "rb") as f:
    item_to_name = pickle.load(f)

with open(ARTIFACT_DIR / "category_to_vector.pkl", "rb") as f:
    category_to_vector = pickle.load(f)

with open(ARTIFACT_DIR / "date_context_lookup.pkl", "rb") as f:
    date_context_lookup = pickle.load(f)

with open(ARTIFACT_DIR / "customer_default_timeslot.pkl", "rb") as f:
    customer_default_timeslot = pickle.load(f)

print("All artifacts loaded.")

All artifacts loaded.


In [4]:
# =========================================
# C4. HELPERS
# =========================================
def normalize_text(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_category(cat):
    if pd.isna(cat):
        return cat
    cat = normalize_text(cat)
    parts = [p.strip() for p in cat.split("-")]
    parts = [p.capitalize() if p else p for p in parts]
    return "-".join(parts)

def infer_season_from_month(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4]:
        return "Spring"
    elif month in [5, 6, 7]:
        return "Summer"
    elif month in [8, 9, 10]:
        return "Autumn"
    else:
        return "LateAutumn"

def week_of_month(dt):
    return ((dt.day - 1) // 7) + 1

def cosine_sim(vec1, vec2):
    if vec1 is None or vec2 is None:
        return 0.0

    v1 = np.array(vec1, dtype=float).reshape(1, -1)
    v2 = np.array(vec2, dtype=float).reshape(1, -1)

    if np.allclose(v1, 0) or np.allclose(v2, 0):
        return 0.0

    return float(cosine_similarity(v1, v2)[0][0])

def mean_pool_vectors(vectors):
    valid = [np.array(v, dtype=float) for v in vectors if v is not None]
    if len(valid) == 0:
        return None
    return np.mean(valid, axis=0)

def build_context_embedding_from_items(item_ids):
    cats = []
    for iid in item_ids:
        cat = item_to_category.get(int(iid), None)
        if cat is not None:
            cats.append(cat)

    cats = list(dict.fromkeys(cats))
    vecs = [category_to_vector.get(cat, None) for cat in cats]
    return mean_pool_vectors(vecs)

In [5]:
# =========================================
# C5. STAGE 1 CANDIDATE GENERATORS
# =========================================
def get_rule_candidates(context_item_ids, rules_df, top_n=30):
    context_set = set([str(int(x)) for x in context_item_ids])
    score_counter = Counter()

    for _, row in rules_df.iterrows():
        antecedents = set(row["antecedents"])
        consequents = [int(x) for x in row["consequents"]]

        if antecedents.issubset(context_set):
            score = float(row["confidence"]) * float(row["lift"])
            for item in consequents:
                score_counter[item] += score

    return [item for item, _ in score_counter.most_common(top_n)]

def get_copurchase_candidates(context_item_ids, item_pair_counter, top_n=30):
    score_counter = Counter()

    for ctx_item in context_item_ids:
        for (a, b), cnt in item_pair_counter.items():
            if b == int(ctx_item):
                score_counter[a] += cnt

    return [item for item, _ in score_counter.most_common(top_n)]

def get_context_candidates(context_key, context_item_counter, top_n=30):
    score_counter = Counter()

    for (ctx_key, item_id), cnt in context_item_counter.items():
        if ctx_key == context_key:
            score_counter[item_id] += cnt

    return [item for item, _ in score_counter.most_common(top_n)]

def get_user_history_candidates(customer_id, user_item_counter, top_n=30):
    score_counter = Counter()

    for (uid, item_id), cnt in user_item_counter.items():
        if uid == customer_id:
            score_counter[item_id] += cnt

    return [item for item, _ in score_counter.most_common(top_n)]

In [6]:
# =========================================
# C6. FEATURE BUILDERS
# =========================================
def get_item_cooccurrence_score(candidate_item_id, context_item_ids):
    if len(context_item_ids) == 0:
        return 0.0

    scores = [item_pair_counter.get((int(candidate_item_id), int(ctx)), 0) for ctx in context_item_ids]
    return float(np.mean(scores)) if len(scores) > 0 else 0.0

def get_customer_history_score(customer_id, candidate_item_id):
    return float(user_item_counter.get((int(customer_id), int(candidate_item_id)), 0))

def get_category_affinity_score(customer_id, candidate_item_id):
    cat = item_to_category.get(int(candidate_item_id), None)
    if cat is None:
        return 0.0
    return float(user_category_counter.get((int(customer_id), cat), 0))

def get_context_popularity_score(context_key, candidate_item_id):
    return float(context_item_counter.get((context_key, int(candidate_item_id)), 0))

def get_embedding_similarity_score(context_embedding_vec, candidate_item_id):
    cat = item_to_category.get(int(candidate_item_id), None)
    cand_vec = category_to_vector.get(cat, None) if cat is not None else None
    return cosine_sim(context_embedding_vec, cand_vec)

In [7]:
# =========================================
# C7. REQUEST CONTEXT BUILDER
# =========================================
def build_request_context(customer_id, date_str):
    dt = pd.to_datetime(date_str, errors="coerce")

    if pd.isna(dt):
        raise ValueError("date parse করা যায়নি")

    date_key = str(dt.date())

    if date_key in date_context_lookup:
        season = date_context_lookup[date_key]["season"]
        is_holiday = date_context_lookup[date_key]["isHoliday"]
        is_festival = date_context_lookup[date_key]["isFestival"]
    else:
        season = infer_season_from_month(dt.month)
        is_holiday = 0
        is_festival = 0

    time_slot = customer_default_timeslot.get(int(customer_id), "Afternoon")
    wom = week_of_month(dt)

    return {
        "date": dt,
        "season": season,
        "isHoliday": int(is_holiday),
        "isFestival": int(is_festival),
        "timeSlot": time_slot,
        "weekOfMonth": int(wom)
    }

In [8]:
# =========================================
# C8. BUILD FEATURE ROWS FOR CANDIDATES
# =========================================
def build_scoring_rows(customer_id, request_context, input_items, candidate_pool):
    context_item_ids = [int(x["item_id"]) for x in input_items]
    context_embedding_vec = build_context_embedding_from_items(context_item_ids)

    context_key = (
        request_context["season"],
        int(request_context["isHoliday"]),
        int(request_context["isFestival"]),
        request_context["timeSlot"],
        int(request_context["weekOfMonth"])
    )

    rows = []

    for cand in candidate_pool:
        cand_cat = item_to_category.get(int(cand), "Unknown")

        rows.append({
            "customerId": int(customer_id),
            "candidate_item_id": int(cand),
            "candidate_category": cand_cat,
            "basket_size": len(context_item_ids),
            "item_cooccurrence_score": get_item_cooccurrence_score(cand, context_item_ids),
            "category_affinity_score": get_category_affinity_score(customer_id, cand),
            "context_popularity_score": get_context_popularity_score(context_key, cand),
            "customer_history_score": get_customer_history_score(customer_id, cand),
            "embedding_similarity_score": get_embedding_similarity_score(context_embedding_vec, cand),
            "season": request_context["season"],
            "isHoliday": int(request_context["isHoliday"]),
            "isFestival": int(request_context["isFestival"]),
            "timeSlot": request_context["timeSlot"],
            "weekOfMonth": int(request_context["weekOfMonth"])
        })

    return pd.DataFrame(rows)

In [9]:
# =========================================
# C9. STAGE 1 PLUS STAGE 2 PIPELINE
# =========================================
def recommend_items(customer_id, date_str, items, top_n=10):
    input_item_ids = [int(x["item_id"]) for x in items]

    request_context = build_request_context(customer_id, date_str)

    context_key = (
        request_context["season"],
        int(request_context["isHoliday"]),
        int(request_context["isFestival"]),
        request_context["timeSlot"],
        int(request_context["weekOfMonth"])
    )

    # ---------- Stage 1 ----------
    rule_candidates = get_rule_candidates(input_item_ids, rules_df, top_n=30)
    copurchase_candidates = get_copurchase_candidates(input_item_ids, item_pair_counter, top_n=30)
    context_candidates = get_context_candidates(context_key, context_item_counter, top_n=30)
    user_candidates = get_user_history_candidates(customer_id, user_item_counter, top_n=30)

    merged_candidates = []
    seen = set()

    for item in rule_candidates + copurchase_candidates + context_candidates + user_candidates:
        if item not in seen and item not in input_item_ids:
            seen.add(item)
            merged_candidates.append(item)

    # ---------- Stage 2 ----------
    score_df = build_scoring_rows(customer_id, request_context, items, merged_candidates)

    if score_df.empty:
        return {
            "ordered_items": [item_to_name.get(int(x["item_id"]), f"Item_{x['item_id']}") for x in items],
            "recommendations": [],
            "stage1_candidate_count": 0
        }

    X_live = score_df[
        [
            "basket_size",
            "item_cooccurrence_score",
            "category_affinity_score",
            "context_popularity_score",
            "customer_history_score",
            "embedding_similarity_score",
            "isHoliday",
            "isFestival",
            "weekOfMonth",
            "season",
            "timeSlot",
            "candidate_category"
        ]
    ].copy()

    X_live = pd.get_dummies(X_live, columns=["season", "timeSlot", "candidate_category"])
    X_live = X_live.reindex(columns=trained_feature_columns, fill_value=0)

    pred_scores = ranker.predict(X_live)
    score_df["score"] = pred_scores
    score_df["itemName"] = score_df["candidate_item_id"].apply(
        lambda x: item_to_name.get(int(x), f"Item_{x}")
    )

    score_df = score_df.sort_values("score", ascending=False).reset_index(drop=True)
    top_df = score_df.head(top_n).copy()

    ordered_items = []
    for x in items:
        ordered_items.append({
            "item_id": int(x["item_id"]),
            "quantity": float(x["quantity"]),
            "item_name": item_to_name.get(int(x["item_id"]), f"Item_{x['item_id']}")
        })

    recs = []
    for _, row in top_df.iterrows():
        recs.append({
            "item_id": int(row["candidate_item_id"]),
            "item_name": row["itemName"],
            "category": row["candidate_category"],
            "score": round(float(row["score"]), 6)
        })

    return {
        "ordered_items": ordered_items,
        "recommendations": recs,
        "stage1_candidate_count": len(merged_candidates),
        "request_context": request_context
    }

In [10]:
# =========================================
# C10. EXAMPLE INPUT
# =========================================
request_input = {
    "customer_id": 23561,
    "date": "2025-01-01",
    "items": [
        {"item_id": 7075, "quantity": 1},
        {"item_id": 27, "quantity": 4},
        {"item_id": 6814, "quantity": 1}
    ]
}

result = recommend_items(
    customer_id=request_input["customer_id"],
    date_str=request_input["date"],
    items=request_input["items"],
    top_n=10
)

print(json.dumps(result, indent=2, default=str))

{
  "ordered_items": [
    {
      "item_id": 7075,
      "quantity": 1.0,
      "item_name": "Herman Peanut Butter-340gm"
    },
    {
      "item_id": 27,
      "quantity": 4.0,
      "item_name": "Pran Toast-250g"
    },
    {
      "item_id": 6814,
      "quantity": 1.0,
      "item_name": "Horlicks Classic Malt-1Kg Jar"
    }
  ],
  "recommendations": [
    {
      "item_id": 16240,
      "item_name": "Polar Cake - 1Ltr",
      "category": "Bakery-Bread",
      "score": 0.267554
    },
    {
      "item_id": 604,
      "item_name": "Cassia Leaf-100gm (Tespata)",
      "category": "Spices-Cooking",
      "score": 0.247834
    },
    {
      "item_id": 30627,
      "item_name": "Bashundhara Atta-2kg",
      "category": "Pantry-Flour",
      "score": 0.183548
    },
    {
      "item_id": 13921,
      "item_name": "Rui Fish-M-(MNK)",
      "category": "Fish-Fresh",
      "score": 0.084449
    },
    {
      "item_id": 1201,
      "item_name": "Fruits Skittles Cho -45g",
      "catego

In [11]:
# =========================================
# C11. NICE DISPLAY
# =========================================
print("===== Ordered Items =====")
for x in result["ordered_items"]:
    print(f"item_id={x['item_id']} | qty={x['quantity']} | name={x['item_name']}")

print("\n===== Request Context =====")
print(result["request_context"])

print("\n===== Top 10 Recommendations =====")
for i, rec in enumerate(result["recommendations"], 1):
    print(
        f"{i}. item_id={rec['item_id']} | name={rec['item_name']} | "
        f"category={rec['category']} | score={rec['score']}"
    )

print("\nStage 1 candidate count:", result["stage1_candidate_count"])

===== Ordered Items =====
item_id=7075 | qty=1.0 | name=Herman Peanut Butter-340gm
item_id=27 | qty=4.0 | name=Pran Toast-250g
item_id=6814 | qty=1.0 | name=Horlicks Classic Malt-1Kg Jar

===== Request Context =====
{'date': Timestamp('2025-01-01 00:00:00'), 'season': 'Winter', 'isHoliday': 0, 'isFestival': 0, 'timeSlot': 'Morning', 'weekOfMonth': 1}

===== Top 10 Recommendations =====
1. item_id=16240 | name=Polar Cake - 1Ltr | category=Bakery-Bread | score=0.267554
2. item_id=604 | name=Cassia Leaf-100gm (Tespata) | category=Spices-Cooking | score=0.247834
3. item_id=30627 | name=Bashundhara Atta-2kg | category=Pantry-Flour | score=0.183548
4. item_id=13921 | name=Rui Fish-M-(MNK) | category=Fish-Fresh | score=0.084449
5. item_id=1201 | name=Fruits Skittles Cho -45g | category=Chocolates-Sweets | score=-0.022078
6. item_id=3182 | name=Starship Chocolate Milk-200ml | category=Dairy-Milk | score=-0.042021
7. item_id=30743 | name=Senora S.Nap Eco(Belt)-15s Pad | category=Personal-Care-S